## Option 1: XAI via SDGRS XAI CLI

* **Input** (`--input-path`): csv abstracts in the format of ZO_up
* **Output** (`--output-path`): csv abstracts with token_scores column (n_tokens x n_labels) that contain the feature attribution values of a selected XAI method

In [1]:
!sdgrs-xai-cli \
    --model-family=scibert \
    --model-path=$HOME/.cache/huggingface/checkpoints/sdg-scibert/allenai/scibert_scivocab_cased-zo_up/checkpoint-553 \
    --method=shap-partition \
    --input-path ../experiments/data/zo_up.csv \
    --output-path ./zo_up_scibert_xai.csv \
    explain

2025-04-28 04:24:42 [info     ] Parsed arguments, loading model and tokenizer...
2025-04-28 04:24:42 [info     ] Loaded model and tokenizer     model_family=scibert
2025-04-28 04:24:42 [info     ] Data manager set up            data_source_and_target=local
2025-04-28 04:24:42 [info     ] Starting explain action        method=shap-partition
2025-04-28 04:24:46 [info     ] Starting to process documents 
2025-04-28 04:24:46 [info     ] Setting up worker              device=device(type='cuda', index=0) real_gpu_id=0 worker_id=0
100%|██████████████████████████████| 10/10 [00:17<00:00,  1.76s/it, in_queue=10]
2025-04-28 04:25:08 [info     ] Finished processing documents  total_processed=10
2025-04-28 04:25:08 [info     ] Exiting...                    


# Option 2: XAI via the SDGRS XAI lib

In [13]:
import torch
from sdgrs_xai.models import setup_model_and_tokenizer
from sdgrs_xai.explainers import get_explainer
from pathlib import Path

XAI_METHOD = "shap-partition" # one of app.explainers.model.ExplainerMethod
MODEL_FAMILY = "scibert"
MODEL_PATH = Path.home() / ".cache/huggingface/checkpoints/sdg-scibert/allenai/scibert_scivocab_cased-zo_up/checkpoint-553"

model, tokenizer = setup_model_and_tokenizer(
    model_family=MODEL_FAMILY,
    method=XAI_METHOD,
    model_path=MODEL_PATH
)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

explainer = get_explainer(
    method_name=XAI_METHOD,
    model=model,
    tokenizer=tokenizer,
    device=device,
    model_family=MODEL_FAMILY,
)

In [14]:
from sdgrs_xai.explainers.model import XAIOutput

xai_output: XAIOutput = explainer.explain(abstract="Is this about clean energy?")
print(xai_output)

XAIOutput(text='Is this about clean energy?', input_tokens=['', 'Is ', 'this ', 'about ', 'clean ', 'energy', '?', ''], token_scores=[[-4.511093720793724e-10, -7.188646122813225e-09, -3.3178366720676422e-09, 4.511093720793724e-10, -3.7834979593753815e-10, -1.9208528101444244e-09, -2.0489096641540527e-08, 1.1350493878126144e-09, 1.1408701539039612e-08, 1.6792910173535347e-08, -5.413312464952469e-09, 1.3504177331924438e-08, 0.0, 2.2118911147117615e-09, -8.280039764940739e-09, -2.1100277081131935e-09, -1.1641532182693481e-09], [-0.0028277831588638946, -0.02032333574607037, -0.0009695390472188592, -0.006707531705615111, 0.009875031333649531, -0.0064503029570914805, 0.1901185647584498, 0.0018059802532661706, 0.024360976880416274, -0.037657080771168694, 0.014586397039238364, -0.13892699917778373, 0.0026522051775828004, -0.0047355867573060095, -0.018255764865898527, -0.006758187679224648, 0.0002129872445948422], [0.001009959276416339, -0.002859593281755224, -0.007475861290004104, 0.0078831705

In [15]:
# xai_output.token_scores are of shape [n_tokens, n_labels]
# therefore this score corresponds to the token "energy"
xai_output.token_scores[5][xai_output.predicted_id]


0.20633853721665218